# Experiment 60 - Taxon Dual Power

**Building on:** exp59 (OOF cmAP=0.95737), the current best OOF.

**Hypothesis:** exp59 uses one file-confidence power for event and texture species. Since texture species are still the bottleneck, test separate power_event and power_texture values around the exp59 optimum.


In [ ]:
import json
from pathlib import Path

nb_dir = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
exp46_path = nb_dir / 'exp46_no_proto.ipynb'
exp46_nb = json.loads(exp46_path.read_text())

def exp46_source(cell_idx):
    src = ''.join(exp46_nb['cells'][cell_idx]['source'])
    if cell_idx == 1:
        src = src.replace("PROJECT_ROOT = Path('/Users/jjannik/Development/kaggle/birdclef')",
                          "PROJECT_ROOT = Path.cwd()\nif PROJECT_ROOT.name == 'notebooks':\n    PROJECT_ROOT = PROJECT_ROOT.parent")
    return src

for cell_idx in range(1, 10):
    print(f'--- executing exp46 cell {cell_idx} ---')
    exec(compile(exp46_source(cell_idx), f'exp46_cell_{cell_idx}', 'exec'), globals())


In [ ]:
# Exp60: separate file-confidence power for event and texture species

def file_confidence_scale_dual(probs, power_event=1.2, power_texture=1.2, n_windows=N_WINDOWS, top_k=2):
    view = probs.reshape(-1, n_windows, probs.shape[1])
    top_mean = np.sort(view, axis=1)[:, -top_k:, :].mean(axis=1, keepdims=True)
    powers = np.where(TEXTURE_MASK, power_texture, power_event).astype(np.float32)
    return (view * np.power(top_mean + 1e-8, powers[None, None, :])).reshape(probs.shape)

def postprocess_taxon_dual_power(logits, power_event=1.2, power_texture=1.2,
                                 alpha_event=0.10, alpha_texture=0.30):
    p = sigmoid(logits / temperatures[None, :])
    p = file_confidence_scale_dual(p, power_event=power_event, power_texture=power_texture)
    p = adaptive_delta_smooth_taxon(p, alpha_event=alpha_event, alpha_texture=alpha_texture)
    return np.clip(p, 0.0, 1.0)

LAM_EVENT = 2.2
LAM_TEXTURE = 9.0
TTA_W0S = [0.85, 0.90, 0.95]
POWER_EVENTS = [0.9, 1.1, 1.2, 1.3, 1.5]
POWER_TEXTURES = [0.8, 1.0, 1.2, 1.4, 1.6]
ALPHA_TEXTURES = [0.25, 0.30, 0.35]
ENSEMBLE_GRIDS = [
    (0.02, 0.56, 0.42),  # exp59 best
    (0.02, 0.58, 0.40),
    (0.02, 0.54, 0.44),
]

prior_0 = apply_prior_taxon(oof_raw_0, oof_meta_sites, oof_meta_hours, prior_tables,
                            lambda_event=LAM_EVENT, lambda_texture=LAM_TEXTURE)
prior_250 = apply_prior_taxon(oof_raw_250, oof_meta_sites, oof_meta_hours, prior_tables,
                              lambda_event=LAM_EVENT, lambda_texture=LAM_TEXTURE)

best_cmap = -1.0
best_cfg = {}
results = []

for wp, wm, wpr in ENSEMBLE_GRIDS:
    fp_0 = wp * oof_proto_0 + wm * oof_mlp_0 + wpr * prior_0
    fp_250 = wp * oof_proto_250 + wm * oof_mlp_250 + wpr * prior_250
    for tta_w0 in TTA_W0S:
        fp_avg = tta_w0 * fp_0 + (1.0 - tta_w0) * fp_250
        for power_event in POWER_EVENTS:
            for power_texture in POWER_TEXTURES:
                for alpha_texture in ALPHA_TEXTURES:
                    probs = postprocess_taxon_dual_power(
                        fp_avg, power_event=power_event, power_texture=power_texture,
                        alpha_event=0.10, alpha_texture=alpha_texture)
                    cmap = padded_cmap(Y_FULL_aligned, probs)
                    auc = macro_auc(Y_FULL_aligned, probs)
                    row = {'tta_w0': tta_w0, 'power_event': power_event, 'power_texture': power_texture,
                           'alpha_texture': alpha_texture, 'wp': wp, 'wm': wm, 'wpr': wpr,
                           'auc': auc, 'cmap': cmap}
                    results.append(row)
                    if cmap > best_cmap:
                        best_cmap = cmap
                        best_cfg = row.copy()

df = pd.DataFrame(results).sort_values('cmap', ascending=False).reset_index(drop=True)
print(f'Best OOF cmAP: {best_cmap:.5f}')
print(f'Best config:   {best_cfg}')
print('\nTop 30 configs:')
print(df.head(30).to_string(index=False))
print('\nBest cmAP by power_event:')
print(df.groupby('power_event')['cmap'].max().reset_index().to_string(index=False))
print('\nBest cmAP by power_texture:')
print(df.groupby('power_texture')['cmap'].max().reset_index().to_string(index=False))

fp_0_f = best_cfg['wp'] * oof_proto_0 + best_cfg['wm'] * oof_mlp_0 + best_cfg['wpr'] * prior_0
fp_250_f = best_cfg['wp'] * oof_proto_250 + best_cfg['wm'] * oof_mlp_250 + best_cfg['wpr'] * prior_250
fp_f = best_cfg['tta_w0'] * fp_0_f + (1.0 - best_cfg['tta_w0']) * fp_250_f
p_best = postprocess_taxon_dual_power(fp_f, power_event=best_cfg['power_event'],
                                      power_texture=best_cfg['power_texture'],
                                      alpha_event=0.10,
                                      alpha_texture=best_cfg['alpha_texture'])
auc_b = macro_auc(Y_FULL_aligned, p_best)
cmap_b = padded_cmap(Y_FULL_aligned, p_best)
cmap_tex = padded_cmap(Y_FULL_aligned[:, TEXTURE_MASK], p_best[:, TEXTURE_MASK])
cmap_evt = padded_cmap(Y_FULL_aligned[:, ~TEXTURE_MASK], p_best[:, ~TEXTURE_MASK])

print('=' * 60)
print('Experiment 60 - Taxon dual power')
print(f"Best: tta_w0={best_cfg['tta_w0']}, power_event={best_cfg['power_event']}, power_texture={best_cfg['power_texture']}, alpha_texture={best_cfg['alpha_texture']}")
print(f"Weights: wp={best_cfg['wp']}, wm={best_cfg['wm']}, wpr={best_cfg['wpr']}")
print(f'AUC={auc_b:.5f}  cmAP={cmap_b:.5f}')
print(f'Texture cmAP={cmap_tex:.5f}  Event cmAP={cmap_evt:.5f}')
print(f'vs exp59 OOF (0.95737): delta cmAP = {cmap_b - 0.95737:+.5f}')
print(f'vs exp46 OOF (0.95579): delta cmAP = {cmap_b - 0.95579:+.5f}')
print(f'Total wall time: {(time.time() - _WALL_START)/60:.1f} min')
